In [7]:
import cv2
import numpy as np
import os

def auto_slice_atlas(atlas_path, output_dir):
    print(f"Loading atlas from: {atlas_path}")
    
    # Load the image preserving the alpha channel (transparency)
    atlas = cv2.imread(atlas_path, cv2.IMREAD_UNCHANGED)
    
    if atlas is None:
        print("ERROR: Could not load the atlas. Check the file name.")
        return
        
    if atlas.shape[2] != 4:
        print("ERROR: Image does not have an alpha channel! This script requires a transparent background.")
        return

    os.makedirs(output_dir, exist_ok=True)

    # Extract just the Alpha channel (Layer 3)
    # 0 = Fully transparent, 255 = Fully opaque
    alpha_channel = atlas[:, :, 3]

    # Create a binary mask where anything not completely transparent is marked as white (255)
    _, mask = cv2.threshold(alpha_channel, 1, 255, cv2.THRESH_BINARY)

    # Find the contours (the exact shapes) of these white blobs
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    print(f"Found {len(contours)} distinct elements on the atlas.")
    print("Slicing and saving...")

    # For visualization, we'll draw rectangles on a copy of the original
    preview = atlas.copy()

    valid_crops = 0
    
    for i, cnt in enumerate(contours):
        # Get the perfect bounding box for the shape
        x, y, w, h = cv2.boundingRect(cnt)
        
        # Filter out tiny specks of dust/noise (less than 3x3 pixels)
        if w < 3 or h < 3:
            continue
            
        # Crop the image using the exact coordinates
        cropped_piece = atlas[y:y+h, x:x+w]
        
        # Save it
        filename = os.path.join(output_dir, f"sprite_{valid_crops:03d}.png")
        cv2.imwrite(filename, cropped_piece)
        
        # Draw a red box on the preview image (BGR format, so red is [0, 0, 255])
        # We write to the first 3 channels, ignoring alpha for the rectangle
        cv2.rectangle(preview, (x, y), (x+w, y+h), (0, 0, 255, 255), 1)
        
        valid_crops += 1

    print(f"Successfully saved {valid_crops} perfectly cropped sprites to the '{output_dir}' folder.")
    
    # Save a preview image so you can easily see what was sliced
    cv2.imwrite("slice_preview_map.png", preview)
    print("Saved 'slice_preview_map.png' to help you identify the sprites.")

if __name__ == "__main__":
    auto_slice_atlas("spotlighthealthbaratlas.png", "perfect_crops")

Loading atlas from: spotlighthealthbaratlas.png
Found 146 distinct elements on the atlas.
Slicing and saving...
Successfully saved 132 perfectly cropped sprites to the 'perfect_crops' folder.
Saved 'slice_preview_map.png' to help you identify the sprites.


In [6]:
import cv2
import numpy as np
from numba import jit

# --- Multi-Class Configuration ---
# You will need to measure the exact offsets for your Turret/Jungle frames
# just like you did for the Champion frame.
TARGET_CLASSES = {
    "Champion": {
        "template_path": "ui_assets/frames/champion_frame.png",
        "threshold": 0.10,  # SQDIFF_NORMED threshold (lower is stricter)
        "hp_offset_x": 26,
        "hp_offset_y": 4,
        "hp_height": 13,
        "mana_offset_y": 17,
        "mana_height": 4,
        "has_mana": True
    },
    "Turret": {
        "template_path": "ui_assets/frames/turret_frame.png",
        "threshold": 0.10,
        "hp_offset_x": 15,   # EXAMPLE VALUE: Change based on your atlas slice
        "hp_offset_y": 5,    # EXAMPLE VALUE
        "hp_height": 10,     # EXAMPLE VALUE
        "mana_offset_y": 0,
        "mana_height": 0,
        "has_mana": False
    }
    # You can easily add "EpicMonster" or "Jungle" here later!
}

# Pre-define color ranges as constants
LOWER_GREEN = np.array([40, 100, 100], dtype=np.uint8)
UPPER_GREEN = np.array([80, 255, 255], dtype=np.uint8)
LOWER_BLUE = np.array([90, 80, 80], dtype=np.uint8)
UPPER_BLUE = np.array([110, 255, 255], dtype=np.uint8)
LOWER_RED1 = np.array([0, 100, 100], dtype=np.uint8)
UPPER_RED1 = np.array([10, 255, 255], dtype=np.uint8)
LOWER_RED2 = np.array([170, 100, 100], dtype=np.uint8)
UPPER_RED2 = np.array([180, 255, 255], dtype=np.uint8)

DENSITY_THRESHOLD = 0.25
GAP_TOLERANCE = 6

@jit(nopython=True)
def calculate_percentage(profile, height, offset, w, threshold=0.25, gap_tolerance=6):
    """Fast percentage calculation using density"""
    density = profile / height
    dense_cols = np.where(density >= threshold)[0]
    
    if len(dense_cols) == 0:
        return 0.0, offset
    
    start_idx = -1
    for i in range(len(dense_cols)):
        if dense_cols[i] >= offset:
            start_idx = i
            break
    
    if start_idx == -1:
        return 0.0, offset
    
    dense_cols = dense_cols[start_idx:]
    
    end = dense_cols[-1]
    if len(dense_cols) > 1:
        for i in range(len(dense_cols) - 1):
            if dense_cols[i+1] - dense_cols[i] > gap_tolerance:
                end = dense_cols[i]
                break
    
    width = end - offset + 1
    full_bar_width = w - offset
    percentage = (width / full_bar_width) * 100.0
    return min(100.0, max(0.0, percentage)), end

def non_max_suppression_multi_class(boxes, overlap_thresh=0.3):
    """Upgraded NMS to handle multiple classes and varying box sizes"""
    if len(boxes) == 0:
        return []
    
    # boxes format: [x1, y1, x2, y2, class_name, w, h]
    boxes_arr = np.array([[b[0], b[1], b[2], b[3]] for b in boxes], dtype=float)
    x1 = boxes_arr[:, 0]
    y1 = boxes_arr[:, 1]
    x2 = boxes_arr[:, 2]
    y2 = boxes_arr[:, 3]
    area = (x2 - x1) * (y2 - y1)
    
    idxs = np.argsort(y1)
    pick = []
    
    while len(idxs) > 0:
        last = len(idxs) - 1
        i = idxs[last]
        pick.append(i)
        
        xx1 = np.maximum(x1[i], x1[idxs[:last]])
        yy1 = np.maximum(y1[i], y1[idxs[:last]])
        xx2 = np.minimum(x2[i], x2[idxs[:last]])
        yy2 = np.minimum(y2[i], y2[idxs[:last]])
        
        w_overlap = np.maximum(0, xx2 - xx1)
        h_overlap = np.maximum(0, yy2 - yy1)
        overlap = (w_overlap * h_overlap) / area[idxs[:last]]
        
        idxs = np.delete(idxs, np.concatenate(([last], np.where(overlap > overlap_thresh)[0])))
    
    return [boxes[i] for i in pick]

def process_healthbar(image, x, y, w, h, class_name):
    """Calculates status dynamically based on the class config"""
    cfg = TARGET_CLASSES[class_name]
    
    y_adj = y + cfg["hp_offset_y"]
    
    # Extract HP Region
    hp_roi = image[y_adj : y_adj + cfg["hp_height"], x : x + w]
    hsv_hp = cv2.cvtColor(hp_roi, cv2.COLOR_BGR2HSV)
    
    # Create HP Masks
    mask_green = cv2.inRange(hsv_hp, LOWER_GREEN, UPPER_GREEN)
    mask_blue = cv2.inRange(hsv_hp, LOWER_BLUE, UPPER_BLUE)
    mask_red = cv2.bitwise_or(
        cv2.inRange(hsv_hp, LOWER_RED1, UPPER_RED1),
        cv2.inRange(hsv_hp, LOWER_RED2, UPPER_RED2)
    )
    
    # Detect Team
    green_px = cv2.countNonZero(mask_green)
    blue_px = cv2.countNonZero(mask_blue)
    red_px = cv2.countNonZero(mask_red)
    
    if blue_px > green_px and blue_px > red_px:
        team, hp_mask, hp_color = 0, mask_blue, (255, 0, 0)  # Ally
    elif green_px > blue_px and green_px > red_px:
        team, hp_mask, hp_color = 1, mask_green, (0, 255, 0)  # Player
    else:
        team, hp_mask, hp_color = 2, mask_red, (0, 0, 255)  # Enemy
    
    # Calculate HP
    hp_profile = np.sum(hp_mask.astype(np.float32), axis=0) / 255.0
    hp_pct, hp_end = calculate_percentage(hp_profile, hp_mask.shape[0], cfg["hp_offset_x"], w, DENSITY_THRESHOLD, GAP_TOLERANCE)
    
    mana_pct, mana_end = 0.0, 0
    
    # Calculate Mana if applicable (Turrets don't have mana)
    if cfg["has_mana"]:
        mana_y_adj = y + cfg["mana_offset_y"]
        mana_roi = image[mana_y_adj : mana_y_adj + cfg["mana_height"], x : x + w]
        hsv_mana = cv2.cvtColor(mana_roi, cv2.COLOR_BGR2HSV)
        mask_mana = cv2.inRange(hsv_mana, LOWER_BLUE, UPPER_BLUE)
        
        mana_profile = np.sum(mask_mana.astype(np.float32), axis=0) / 255.0
        mana_pct, mana_end = calculate_percentage(mana_profile, mask_mana.shape[0], cfg["hp_offset_x"], w, DENSITY_THRESHOLD, GAP_TOLERANCE)
    
    return hp_pct, mana_pct, team, hp_color, x + hp_end, x + mana_end, cfg

def main():
    # 1. Load all templates dynamically
    templates = {}
    for name, cfg in TARGET_CLASSES.items():
        img = cv2.imread(cfg["template_path"], cv2.IMREAD_UNCHANGED)
        if img is not None:
            templates[name] = {
                "rgb": img[:, :, :3],
                "mask": img[:, :, 3] if img.shape[2] == 4 else None,
                "w": img.shape[1],
                "h": img.shape[0]
            }
        else:
            print(f"Warning: Could not load template for {name}")

    # Load video frame
    cap = cv2.VideoCapture('/Users/montesillo/Downloads/Screen Recording 2025-10-29 at 23.05.29.mov')
    cap.set(cv2.CAP_PROP_POS_FRAMES, 14500)
    ret, target_image = cap.read()
    cap.release()

    if not ret:
        return

    target_copy = target_image.copy()
    all_raw_matches = []

    # 2. Sweep the image for EVERY class type
    for class_name, tmpl in templates.items():
        result = cv2.matchTemplate(
            target_image, 
            tmpl["rgb"], 
            cv2.TM_SQDIFF_NORMED, 
            mask=tmpl["mask"]
        )
        
        # Grab matches below the specific threshold for this class
        locations = np.where(result <= TARGET_CLASSES[class_name]["threshold"])
        
        for pt in zip(*locations[::-1]):
            x, y = pt
            w, h = tmpl["w"], tmpl["h"]
            # Save format: [x1, y1, x2, y2, class_name, w, h]
            all_raw_matches.append((x, y, x + w, y + h, class_name, w, h))

    # 3. Apply global NMS so overlapping frames don't double-detect
    filtered_matches = non_max_suppression_multi_class(all_raw_matches)

    # 4. Process and Draw
    for match in filtered_matches:
        x, y, _, _, class_name, w, h = match
        
        hp_pct, mana_pct, team, hp_color, hp_x, mana_x, cfg = process_healthbar(target_image, x, y, w, h, class_name)
        
        y_adj = y + cfg["hp_offset_y"]
        h_adj = cfg["hp_height"]
        
        # Draw visualizations
        cv2.rectangle(target_copy, (x, y_adj), (x + w, y_adj + h_adj), hp_color, 2)
        cv2.line(target_copy, (int(hp_x), y_adj), (int(hp_x), y_adj + h_adj), hp_color, 2)
        
        if cfg["has_mana"]:
            mana_y_adj = y + cfg["mana_offset_y"]
            cv2.line(target_copy, (int(mana_x), mana_y_adj), (int(mana_x), mana_y_adj + cfg["mana_height"]), (255, 0, 0), 2)
        
        cv2.line(target_copy, (x + cfg["hp_offset_x"], y_adj), (x + cfg["hp_offset_x"], y_adj + h_adj), hp_color, 2)

        # Display Team and Class Tag (e.g. "[ENEMY Champion]")
        team_str = ["ALLY", "PLAYER", "ENEMY"][team]
        tag_text = f'[{team_str} {class_name}]'
        
        cv2.putText(target_copy, tag_text, (x + w - 150, y - 5), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, hp_color, 1)      
        cv2.putText(target_copy, f'{hp_pct:.0f}%', (x + w + 5, y_adj + 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, hp_color, 1)
                   
        if cfg["has_mana"]:
            cv2.putText(target_copy, f'{mana_pct:.0f}%', (x + w + 5, y + cfg["mana_offset_y"] + 5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

    # Save and display
    cv2.imwrite('detected_multi_class.png', target_copy)

if __name__ == "__main__":
    main()

[ WARN:0@935.689] global loadsave.cpp:275 findDecoder imread_('ui_assets/frames/champion_frame.png'): can't open/read file: check file path/integrity
[ WARN:0@935.689] global loadsave.cpp:275 findDecoder imread_('ui_assets/frames/turret_frame.png'): can't open/read file: check file path/integrity
OpenCV: Couldn't read video stream from file "/Users/montesillo/Downloads/Screen Recording 2025-10-29 at 23.05.29.mov"
